# 09 — Precompute the AI matches for the app (Google Colab)

The app's default matching, **"AI + team check"**, needs the LLM answer for each of the 48 recipe ingredients.
The LLM needs a GPU, the demo laptop does not have one, so we compute the answers **once** here and save them to
`data/ingredient_matches_llm.csv`. The app reads that file.

**Configuration:** `Qwen2.5-14B-Instruct` + verify step, threshold 0.3 — the one selected on TUNE in notebook 08.
This notebook does **not** evaluate anything (no TEST labels are used); it only produces the AI answers.

**Before running:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all* (about 15–20 minutes).
If Colab says there is no GPU available today, wait a few hours (free daily limit).

In [1]:
# 1. GPU check (should show a Tesla T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
# 2. Code and packages
!git clone -q -b llm-matcher https://github.com/laurarigau/Food-Sustainability.git
%cd Food-Sustainability
!pip install -q -r requirements.txt
!pip install -q transformers accelerate bitsandbytes

/content/Food-Sustainability


In [3]:
# 3. Run the LLM once on the 48 recipe ingredients
!python src/precompute_matches.py --model Qwen/Qwen2.5-14B-Instruct --verify

config.json: 100% 663/663 [00:00<00:00, 3.49MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 23.0MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 65.8MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 116MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 141MB/s]
model.safetensors.index.json: 100% 47.5k/47.5k [00:00<00:00, 118MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0% 0/8 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/4.00G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/7.99G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/16.0G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/25.6G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/29.5G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/29.5G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/29.5G [00:00<?, ?B/s]
Reconstructing (incomplete total

In [4]:
# 4. Quick look: how often does the AI agree with the team?
import pandas as pd
ai = pd.read_csv("data/ingredient_matches_llm.csv", dtype={"agribalyse_code": str})
labels = pd.read_csv("data/ingredient_labels.csv", dtype={"agribalyse_code": str}).drop_duplicates("canonical_ingredient")
m = ai.merge(labels[["canonical_ingredient", "agribalyse_code", "agribalyse_name"]],
             left_on="ingredient", right_on="canonical_ingredient", suffixes=("_ai", "_team"))
m["agrees"] = (~m.uncertain) & (m.agribalyse_code_ai == m.agribalyse_code_team)
print(f"AI agrees with the team on {m.agrees.sum()}/{len(m)} recipe ingredients; not sure on {m.uncertain.sum()}")
print(m.loc[~m.agrees, ["ingredient", "agribalyse_name_ai", "agribalyse_name_team", "uncertain"]].to_string(index=False))

AI agrees with the team on 34/48 recipe ingredients; not sure on 5
 ingredient               agribalyse_name_ai                    agribalyse_name_team  uncertain
       beef                   Beef, rib, raw        Beef, minced steak, 15% fat, raw      False
bell pepper                              NaN Sweet pepper, green, yellow or red, raw       True
 black bean    Haricot bean, canned, drained           Haricot bean, canned, drained       True
    cabbage                 Red cabbage, raw                      White cabbage, raw       True
    chicken Chicken leg, meat, roasted/baked      Chicken, breast, without skin, raw      False
  green pea                 Garden peas, raw                Garden peas, frozen, raw      False
       lamb                Lamb, saddle, raw                     Lamb, shoulder, raw      False
      lemon          Lemon juice, pure juice                        Lemon, pulp, raw       True
     lentil                    Lentil, dried                         

In [5]:
# 5. Download the file and put it in the data/ folder of the repo
from google.colab import files
files.download("data/ingredient_matches_llm.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>